# 02 — GAN 2D Unnormalized
**Dự án:** Latent Manipulation of Brain MRI using Volume-Preserving GANs

**Input:** PNG unnormalized từ `00a_preprocessing_2d.ipynb`

**Kiến trúc:**
- **Generator**: U-Net + Age Embedding inject vào bottleneck
- **Discriminator**: PatchGAN + Age Conditioning

**Output:**
```
gan2d_unnormalized/
└── best_model.pth   ← checkpoint tốt nhất (loss_G thấp nhất)
```

## Bước 1: Cấu hình

In [1]:
import os

# ==== ĐƯỜNG DẪN ====
# Khác file 01: trỏ tới unnormalized thay vì normalized
DATA_DIR   = '/kaggle/input/datasets/minhbodoi/full-preprocessed-2d/preprocessed_2d/unnormalized'
LABELS_CSV = '/kaggle/input/datasets/minhbodoi/full-preprocessed-2d/preprocessed_2d/preprocessing_log.csv'
OUTPUT_DIR = '/kaggle/working/gan2d_unnormalized'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==== HYPERPARAMETERS ====
IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_EPOCHS = 500  # 2D: 500 epoch
PATIENCE   = 20   # dừng nếu val_SSIM không tăng >= MIN_DELTA sau 20 epoch liên tiếp
LR_G       = 2e-4
LR_D       = 2e-4
LAMBDA_L1  = 10   # giảm từ 100 → 10 để model dám thay đổi tuổi
LAMBDA_AGE = 5    # tăng từ 1 → 5 để age loss có trọng lượng hơn
LAMBDA_GP  = 10   # gradient penalty cho WGAN-GP
LATENT_DIM = 256

print(f'Data dir : {DATA_DIR}')
print(f'PNG files: {len([f for f in os.listdir(DATA_DIR) if f.endswith(".png")])}')

Data dir : /kaggle/input/datasets/minhbodoi/full-preprocessed-2d/preprocessed_2d/unnormalized
PNG files: 12497


## Bước 2: Import thư viện

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU : Tesla T4
VRAM: 15.6 GB


## Bước 3: Dataset

In [3]:
from torch.utils.data import random_split

class BrainMRI2DDataset(Dataset):
    def __init__(self, data_dir, labels_csv, image_size=256):
        self.data_dir = data_dir
        df = pd.read_csv(labels_csv)
        df['full_path'] = df['png_file'].apply(lambda x: os.path.join(data_dir, x))
        df = df[df['full_path'].apply(os.path.exists)].reset_index(drop=True)
        self.df      = df
        self.age_min = df['age'].min()
        self.age_max = df['age'].max()
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
        print(f'Dataset: {len(df)} ảnh | tuổi [{self.age_min:.1f}, {self.age_max:.1f}]')

    def normalize_age(self, age):
        return 2 * (age - self.age_min) / (self.age_max - self.age_min) - 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img      = Image.open(row['full_path']).convert('L')
        img      = self.transform(img)
        age_norm = torch.tensor(self.normalize_age(row['age']), dtype=torch.float32)
        age_raw  = torch.tensor(row['age'] / 100.0, dtype=torch.float32)
        return img, age_norm, age_raw


# ── Train / Test split 80/20 ──────────────────────────────────
full_dataset = BrainMRI2DDataset(DATA_DIR, LABELS_CSV, IMAGE_SIZE)
n_total  = len(full_dataset)
n_train  = int(0.8 * n_total)
n_test   = n_total - n_train

train_set, test_set = random_split(
    full_dataset, [n_train, n_test],
    generator=torch.Generator().manual_seed(42)  # seed cố định để tái lặp
)

# Lưu lại age_min/max từ full dataset (không tính riêng train)
dataset = full_dataset  # dùng để lấy age_min/max khi save checkpoint

dataloader      = DataLoader(train_set, batch_size=BATCH_SIZE,
                             shuffle=True,  num_workers=4, pin_memory=True)
dataloader_test = DataLoader(test_set,  batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=4, pin_memory=True)

print(f'Train: {n_train} ảnh | Test: {n_test} ảnh')


Dataset: 12497 ảnh | tuổi [18.0, 97.0]
Train: 9997 ảnh | Test: 2500 ảnh


## Bước 4: Kiến trúc Model

In [4]:
class AgeEmbedding(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 128), nn.ReLU(),
            nn.Linear(128, embed_dim)
        )
    def forward(self, age): return self.net(age.unsqueeze(-1))


class MappingNetwork(nn.Module):
    """
    StyleGAN-inspired: age scalar -> w style vector (512-dim).
    Disentangle age từ image features.
    """
    def __init__(self, latent_dim=256, w_dim=512, n_layers=4):
        super().__init__()
        layers = [AgeEmbedding(latent_dim), nn.ReLU()]
        in_dim = latent_dim
        for _ in range(n_layers - 1):
            layers += [nn.Linear(in_dim, w_dim), nn.ReLU()]
            in_dim = w_dim
        self.net = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, w_dim)

    def forward(self, age):
        return self.out(self.net(age))  # (B, w_dim)


class AdaIN2D(nn.Module):
    """
    Adaptive Instance Normalization: normalize feature map,
    rồi scale + shift theo style vector w.
    Cho phép age inject ở mỗi decoder block thay vì chỉ bottleneck.
    """
    def __init__(self, channels, w_dim=512):
        super().__init__()
        self.norm  = nn.InstanceNorm2d(channels, affine=False)
        self.style = nn.Linear(w_dim, channels * 2)  # scale + shift

    def forward(self, x, w):
        style = self.style(w).unsqueeze(-1).unsqueeze(-1)  # (B, 2C, 1, 1)
        scale, shift = style.chunk(2, dim=1)               # (B, C, 1, 1)
        return scale * self.norm(x) + shift


class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, down=True, use_bn=True, dropout=False):
        super().__init__()
        layers = []
        if down: layers.append(nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False))
        else:    layers.append(nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False))
        if use_bn:  layers.append(nn.BatchNorm2d(out_ch))
        if dropout: layers.append(nn.Dropout(0.5))
        layers.append(nn.LeakyReLU(0.2) if down else nn.ReLU())
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class Generator(nn.Module):
    """
    StyleGAN-inspired U-Net Generator.
    - Encoder: extract image features (không có age)
    - MappingNetwork: age -> w style vector
    - Decoder: AdaIN inject age style ở mỗi block
    => Age được disentangle khỏi image content
    """
    def __init__(self, latent_dim=256, w_dim=512):
        super().__init__()
        self.mapping = MappingNetwork(latent_dim, w_dim)

        # Encoder (giữ nguyên, không có age)
        self.e1 = UNetBlock(1,   64,  down=True, use_bn=False)
        self.e2 = UNetBlock(64,  128, down=True)
        self.e3 = UNetBlock(128, 256, down=True)
        self.e4 = UNetBlock(256, 512, down=True)
        self.e5 = UNetBlock(512, 512, down=True)
        self.e6 = UNetBlock(512, 512, down=True)
        self.e7 = UNetBlock(512, 512, down=True)
        self.e8 = UNetBlock(512, 512, down=True, use_bn=False)

        # Decoder blocks (không có BatchNorm — dùng AdaIN thay thế)
        self.d1 = nn.Sequential(nn.ConvTranspose2d(512,  512, 4, 2, 1, bias=False), nn.Dropout(0.5), nn.ReLU())
        self.d2 = nn.Sequential(nn.ConvTranspose2d(1024, 512, 4, 2, 1, bias=False), nn.Dropout(0.5), nn.ReLU())
        self.d3 = nn.Sequential(nn.ConvTranspose2d(1024, 512, 4, 2, 1, bias=False), nn.Dropout(0.5), nn.ReLU())
        self.d4 = nn.Sequential(nn.ConvTranspose2d(1024, 512, 4, 2, 1, bias=False), nn.ReLU())
        self.d5 = nn.Sequential(nn.ConvTranspose2d(1024, 256, 4, 2, 1, bias=False), nn.ReLU())
        self.d6 = nn.Sequential(nn.ConvTranspose2d(512,  128, 4, 2, 1, bias=False), nn.ReLU())
        self.d7 = nn.Sequential(nn.ConvTranspose2d(256,  64,  4, 2, 1, bias=False), nn.ReLU())
        self.out = nn.Sequential(nn.ConvTranspose2d(128, 1, 4, 2, 1), nn.Tanh())

        # AdaIN cho mỗi decoder block
        self.adain1 = AdaIN2D(512,  w_dim)
        self.adain2 = AdaIN2D(512,  w_dim)
        self.adain3 = AdaIN2D(512,  w_dim)
        self.adain4 = AdaIN2D(512,  w_dim)
        self.adain5 = AdaIN2D(256,  w_dim)
        self.adain6 = AdaIN2D(128,  w_dim)
        self.adain7 = AdaIN2D(64,   w_dim)

    def forward(self, x, age):
        w = self.mapping(age)  # (B, w_dim) — style vector từ age

        # Encode image (không có age)
        e1=self.e1(x); e2=self.e2(e1); e3=self.e3(e2); e4=self.e4(e3)
        e5=self.e5(e4); e6=self.e6(e5); e7=self.e7(e6); e8=self.e8(e7)

        # Decode với AdaIN inject age style ở mỗi block
        d1 = self.adain1(self.d1(e8), w)
        d2 = self.adain2(self.d2(torch.cat([d1, e7], 1)), w)
        d3 = self.adain3(self.d3(torch.cat([d2, e6], 1)), w)
        d4 = self.adain4(self.d4(torch.cat([d3, e5], 1)), w)
        d5 = self.adain5(self.d5(torch.cat([d4, e4], 1)), w)
        d6 = self.adain6(self.d6(torch.cat([d5, e3], 1)), w)
        d7 = self.adain7(self.d7(torch.cat([d6, e2], 1)), w)
        return self.out(torch.cat([d7, e1], 1))

    def encode(self, x, age):
        """Trả về w style vector — dùng cho latent manipulation."""
        return self.mapping(age)  # (B, w_dim)

    def decode_from_w(self, x, w):
        """Decode từ w trực tiếp — dùng khi manipulate w."""
        e1=self.e1(x); e2=self.e2(e1); e3=self.e3(e2); e4=self.e4(e3)
        e5=self.e5(e4); e6=self.e6(e5); e7=self.e7(e6); e8=self.e8(e7)
        d1 = self.adain1(self.d1(e8), w)
        d2 = self.adain2(self.d2(torch.cat([d1, e7], 1)), w)
        d3 = self.adain3(self.d3(torch.cat([d2, e6], 1)), w)
        d4 = self.adain4(self.d4(torch.cat([d3, e5], 1)), w)
        d5 = self.adain5(self.d5(torch.cat([d4, e4], 1)), w)
        d6 = self.adain6(self.d6(torch.cat([d5, e3], 1)), w)
        d7 = self.adain7(self.d7(torch.cat([d6, e2], 1)), w)
        return self.out(torch.cat([d7, e1], 1))


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(2, 64, 4, 2, 1, bias=False), nn.LeakyReLU(0.2),
            nn.Conv2d(64,  128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            nn.Conv2d(256, 512, 4, 1, 1, bias=False), nn.BatchNorm2d(512), nn.LeakyReLU(0.2),
            nn.Conv2d(512, 1, 4, 1, 1)
        )
    def forward(self, img, age):
        age_map = age.view(-1, 1, 1, 1).expand(-1, 1, img.shape[2], img.shape[3])
        return self.model(torch.cat([img, age_map], dim=1))


G = Generator(LATENT_DIM).to(DEVICE)
D = Discriminator().to(DEVICE)
print(f'Generator    : {sum(p.numel() for p in G.parameters())/1e6:.1f}M params')
print(f'Discriminator: {sum(p.numel() for p in D.parameters())/1e6:.1f}M params')


Generator    : 54.6M params
Discriminator: 2.8M params


## Bước 5: Loss + Optimizers

Dùng **WGAN-GP** thay vì BCE để tránh mode collapse và training ổn định hơn.

In [5]:
# ── WGAN-GP losses ───────────────────────────────────────────
criterion_l1  = nn.L1Loss()
criterion_age = nn.MSELoss()

def compute_gradient_penalty(D, real, fake, ages, device):
    B = real.size(0)
    alpha = torch.rand(B, 1, 1, 1, device=device)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp = D(interp, ages)
    grad = torch.autograd.grad(
        outputs=d_interp, inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True, retain_graph=True
    )[0]
    gp = ((grad.norm(2, dim=1) - 1) ** 2).mean()
    return gp

# Age regressor trên ảnh fake (giữ nguyên)
age_regressor = nn.Sequential(
    nn.AdaptiveAvgPool2d(8), nn.Flatten(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1), nn.Sigmoid()
).to(DEVICE)

# Age regressor trên w — buộc w phải encode tuổi có cấu trúc
# Input: w (B, 512), Output: tuổi normalize (B, 1)
w_age_regressor = nn.Sequential(
    nn.Linear(512, 128), nn.ReLU(),
    nn.Linear(128, 32),  nn.ReLU(),
    nn.Linear(32, 1),    nn.Sigmoid()
).to(DEVICE)

LAMBDA_W_AGE = 10  # weight cho w age loss — quan trọng để disentangle

opt_G = optim.Adam(
    list(G.parameters()) + list(age_regressor.parameters()) + list(w_age_regressor.parameters()),
    lr=LR_G, betas=(0.0, 0.9)
)
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.0, 0.9))

scheduler_G = optim.lr_scheduler.StepLR(opt_G, step_size=50, gamma=0.5)
scheduler_D = optim.lr_scheduler.StepLR(opt_D, step_size=50, gamma=0.5)

print('Loss + Optimizers (WGAN-GP + StyleGAN AdaIN) sẵn sàng!')


Loss + Optimizers (WGAN-GP) sẵn sàng!


## Bước 6: Training

In [6]:
from skimage.metrics import structural_similarity as ssim_fn
import numpy as np, subprocess, shutil, tempfile

# ── Duong dan checkpoint ──────────────────────────────────────
CKPT_PATH    = f'{OUTPUT_DIR}/last_checkpoint.pth'
BEST_PATH    = f'{OUTPUT_DIR}/best_model.pth'

# ── Kaggle Dataset info ───────────────────────────────────────
_ku = subprocess.run('kaggle config view', shell=True, capture_output=True, text=True).stdout
KAGGLE_USER  = [l.split(':')[1].strip() for l in _ku.split('\n') if 'username' in l][0]
DATASET_NAME = os.path.basename(OUTPUT_DIR).replace('_', '-')
MIN_DELTA    = 0.005  # chi tinh la 'cai thien that' khi SSIM tang >= 0.005

def push_checkpoints_to_kaggle(msg):
    tmp = tempfile.mkdtemp()
    try:
        for fn in ['last_checkpoint.pth', 'best_model.pth']:
            if os.path.exists(f'{OUTPUT_DIR}/{fn}'):
                shutil.copy2(f'{OUTPUT_DIR}/{fn}', f'{tmp}/{fn}')
        import json as _j
        with open(f'{tmp}/dataset-metadata.json', 'w') as _f:
            _j.dump({'title': DATASET_NAME,
                     'id'   : f'{KAGGLE_USER}/{DATASET_NAME}',
                     'licenses': [{'name': 'CC0-1.0'}]}, _f)
        chk = subprocess.run(
            f'kaggle datasets list --user {KAGGLE_USER} --search {DATASET_NAME}',
            shell=True, capture_output=True, text=True)
        if DATASET_NAME in chk.stdout:
            subprocess.run(f'kaggle datasets version -p {tmp} -m "{msg}"', shell=True)
        else:
            subprocess.run(f'kaggle datasets create -p {tmp}', shell=True)
        print(f'  Cloud: {msg} -> {KAGGLE_USER}/{DATASET_NAME}')
    finally:
        shutil.rmtree(tmp)

# ── Resume neu co checkpoint ──────────────────────────────────
start_epoch    = 1
patience_count = 0
best_val_ssim  = -1.0
history = {'loss_G': [], 'loss_D': [], 'loss_L1': [], 'loss_age': [], 'val_ssim': []}

def find_checkpoint(filename):
    p = f'{OUTPUT_DIR}/{filename}'
    if os.path.exists(p): return p, 'working'
    import glob
    matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if matches: return matches[0], 'input'
    return None, None

load_path = None
for fname in ['last_checkpoint.pth', 'best_model.pth']:
    p, src_loc = find_checkpoint(fname)
    if p:
        load_path = p
        print(f'Tim thay {fname} ({src_loc}) — load de train tiep...')
        break
if not load_path:
    print('Khong co checkpoint — train tu dau')

if load_path:
    ckpt = torch.load(load_path, map_location=DEVICE)
    G.load_state_dict(ckpt['G_state'])
    D.load_state_dict(ckpt['D_state'])
    opt_G.load_state_dict(ckpt['opt_G'])
    opt_D.load_state_dict(ckpt['opt_D'])
    start_epoch    = ckpt['epoch'] + 1
    best_val_ssim  = ckpt.get('best_val_ssim', -1.0)
    patience_count = ckpt.get('patience_count', 0)
    history        = ckpt.get('history', history)
    if 'w_age_reg_state' in ckpt:
        w_age_regressor.load_state_dict(ckpt['w_age_reg_state'])
    print(f'Resume tu epoch {ckpt["epoch"]} | best_val_SSIM={best_val_ssim:.4f}')
    print(f'   Tiep tuc tu epoch {start_epoch} -> {NUM_EPOCHS}')

N_CRITIC = 5
print(f'Bat dau training {NUM_EPOCHS} epochs (WGAN-GP)...\n')

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    G.train(); D.train()
    epoch_loss_G = epoch_loss_D = epoch_loss_L1 = epoch_loss_age = 0
    n_batches = 0

    data_iter = iter(dataloader)
    for _ in range(len(dataloader) // N_CRITIC):
        for _ in range(N_CRITIC):
            try:
                real_imgs, ages_norm, ages_raw = next(data_iter)
            except StopIteration:
                data_iter = iter(dataloader)
                real_imgs, ages_norm, ages_raw = next(data_iter)
            real_imgs = real_imgs.to(DEVICE)
            ages_norm = ages_norm.to(DEVICE)
            ages_raw  = ages_raw.to(DEVICE)
            opt_D.zero_grad()
            with torch.no_grad():
                fake_imgs = G(real_imgs, ages_norm)
            loss_D_real = -D(real_imgs, ages_norm).mean()
            loss_D_fake =  D(fake_imgs.detach(), ages_norm).mean()
            gp = compute_gradient_penalty(D, real_imgs, fake_imgs.detach(), ages_norm, DEVICE)
            loss_D = loss_D_real + loss_D_fake + LAMBDA_GP * gp
            loss_D.backward()
            opt_D.step()
            epoch_loss_D += loss_D.item()

        try:
            real_imgs, ages_norm, ages_raw = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            real_imgs, ages_norm, ages_raw = next(data_iter)
        real_imgs = real_imgs.to(DEVICE)
        ages_norm = ages_norm.to(DEVICE)
        ages_raw  = ages_raw.to(DEVICE)
        opt_G.zero_grad()
        fake_imgs  = G(real_imgs, ages_norm)
        loss_G_adv = -D(fake_imgs, ages_norm).mean()
        loss_L1    = criterion_l1(fake_imgs, real_imgs) * LAMBDA_L1
        pred_age   = age_regressor(fake_imgs).squeeze()
        loss_age   = criterion_age(pred_age, ages_raw) * LAMBDA_AGE
        # StyleGAN: w phải predict được tuổi → disentangle
        w          = G.mapping(ages_norm)
        pred_w_age = w_age_regressor(w).squeeze()
        loss_w_age = criterion_age(pred_w_age, ages_norm) * LAMBDA_W_AGE
        loss_G     = loss_G_adv + loss_L1 + loss_age + loss_w_age
        loss_G.backward()
        opt_G.step()
        epoch_loss_G   += loss_G_adv.item()
        epoch_loss_L1  += loss_L1.item()
        epoch_loss_age += loss_age.item()
        n_batches += 1

    scheduler_G.step()
    scheduler_D.step()

    n = max(n_batches, 1)
    avg_loss_G   = epoch_loss_G   / n
    avg_loss_D   = epoch_loss_D   / (n * N_CRITIC)
    avg_loss_L1  = epoch_loss_L1  / n
    avg_loss_age = epoch_loss_age / n

    G.eval()
    ssim_scores = []
    with torch.no_grad():
        for real_imgs, ages_norm, _ in dataloader_test:
            real_imgs = real_imgs.to(DEVICE)
            shuffled_idx  = torch.randperm(len(ages_norm))
            ages_shuffled = ages_norm[shuffled_idx].to(DEVICE)
            fake_imgs = G(real_imgs, ages_shuffled)
            for i in range(real_imgs.size(0)):
                r = (real_imgs[i, 0].cpu().numpy() + 1) / 2
                f = (fake_imgs[i, 0].cpu().numpy() + 1) / 2
                ssim_scores.append(ssim_fn(r, f, data_range=1.0))
    val_ssim = float(np.mean(ssim_scores))

    history['loss_G'].append(avg_loss_G)
    history['loss_D'].append(avg_loss_D)
    history['loss_L1'].append(avg_loss_L1)
    history['loss_age'].append(avg_loss_age)
    history['val_ssim'].append(val_ssim)

    # ── Luu last_checkpoint ───────────────────────────────────
    torch.save({
        'epoch'         : epoch,
        'G_state'       : G.state_dict(),
        'D_state'       : D.state_dict(),
        'opt_G'         : opt_G.state_dict(),
        'opt_D'         : opt_D.state_dict(),
        'history'       : history,
        'age_min'       : dataset.age_min,
        'age_max'       : dataset.age_max,
        'best_val_ssim' : best_val_ssim,
        'patience_count': patience_count,
        'test_indices'       : test_set.indices,
        'w_age_reg_state'    : w_age_regressor.state_dict(),
    }, CKPT_PATH)

    # ── Luu best model neu cai thien ──────────────────────────
    if val_ssim > best_val_ssim + MIN_DELTA:
        best_val_ssim  = val_ssim
        patience_count = 0
        torch.save({
            'epoch'         : epoch,
            'G_state'       : G.state_dict(),
            'D_state'       : D.state_dict(),
            'opt_G'         : opt_G.state_dict(),
            'opt_D'         : opt_D.state_dict(),
            'history'       : history,
            'age_min'       : dataset.age_min,
            'age_max'       : dataset.age_max,
            'best_loss_G'   : avg_loss_G,
            'best_val_ssim' : best_val_ssim,
            'val_ssim'      : val_ssim,
            'test_indices'       : test_set.indices,
            'w_age_reg_state'    : w_age_regressor.state_dict(),
        }, BEST_PATH)
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
              f'loss_G={avg_loss_G:.4f} | loss_D={avg_loss_D:.4f} | '
              f'loss_L1={avg_loss_L1:.4f} | val_SSIM={val_ssim:.4f} | [BEST]')
    else:
        patience_count += 1
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
              f'loss_G={avg_loss_G:.4f} | loss_D={avg_loss_D:.4f} | '
              f'loss_L1={avg_loss_L1:.4f} | val_SSIM={val_ssim:.4f} | '
              f'no improve {patience_count}/{PATIENCE}')

    # ── Push moi epoch ────────────────────────────────────────
    push_checkpoints_to_kaggle(f'{epoch}/{NUM_EPOCHS}')

    # ── Early stopping ────────────────────────────────────────
    if patience_count >= PATIENCE:
        print(f'Early stopping tai epoch {epoch}')
        break

print(f'\n=== TRAINING HOAN THANH ===')
print(f'Best val_SSIM: {best_val_ssim:.4f}')
print(f'Model luu tai: {OUTPUT_DIR}/best_model.pth')


Khong co checkpoint — train tu dau
Bat dau training 500 epochs (WGAN-GP)...

Epoch   1/500 | loss_G=1.4458 | loss_D=1.0124 | loss_L1=1.0559 | val_SSIM=0.8455 | [BEST]
Starting upload for file best_model.pth


100%|██████████| 656M/656M [00:07<00:00, 98.1MB/s]
  2%|▏         | 16.4M/656M [00:00<00:03, 169MB/s]

Upload successful: best_model.pth (656MB)
Starting upload for file last_checkpoint.pth


100%|██████████| 656M/656M [00:08<00:00, 77.5MB/s] 


Upload successful: last_checkpoint.pth (656MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/minhbodoi/gan2d-unnormalized
  Cloud: 1/500 -> minhbodoi/gan2d-unnormalized
Epoch   2/500 | loss_G=4.8896 | loss_D=1.4973 | loss_L1=0.3931 | val_SSIM=0.7191 | no improve 1/20
Starting upload for file best_model.pth


...

Upload successful: last_checkpoint.pth (656MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/minhbodoi/gan2d-unnormalized
  Cloud: 37/500 -> minhbodoi/gan2d-unnormalized
Epoch  38/500 | loss_G=-2.0985 | loss_D=1.8883 | loss_L1=0.2454 | val_SSIM=0.9012 | no improve 20/20
Starting upload for file best_model.pth


100%|██████████| 656M/656M [00:08<00:00, 81.2MB/s] 
  3%|▎         | 21.8M/656M [00:00<00:02, 223MB/s]

Upload successful: best_model.pth (656MB)
Starting upload for file last_checkpoint.pth


100%|██████████| 656M/656M [00:05<00:00, 134MB/s] 


Upload successful: last_checkpoint.pth (656MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/minhbodoi/gan2d-unnormalized
  Cloud: 38/500 -> minhbodoi/gan2d-unnormalized
Early stopping tai epoch 38

=== TRAINING HOAN THANH ===
Best val_SSIM: 0.9671
Model luu tai: /kaggle/working/gan2d_unnormalized/best_model.pth


In [7]:
# Push da duoc tich hop vao training loop (cell tren) — chay moi epoch.
print(f'Checkpoints da duoc push len: {KAGGLE_USER}/{DATASET_NAME}')


Checkpoints da duoc push len: minhbodoi/gan2d-unnormalized
